> **Provenance only — not runnable from the public repository.**
>
> This notebook reproduces the **full-cohort** Table 1 (all rows, n = 105) and
> requires the non-public cleaned cohort file
> `data/BGA_merged_all_20260208_cleaned.csv`, which is **not distributed** with
> this repository (it contains data beyond the released blinded subset). Because
> that input is unavailable publicly, this notebook cannot be executed here and
> its outputs have been cleared.
>
> For the reproducible, public path — the subset of Table 1 derivable from the
> shipped blinded dataset — use **`03_table_1_generation_blinded.ipynb`**.

# 02 — Table 1 Generation

A.L. 2026-03-16 (updated 2026-03-18; 2026-04-13 for manuscript Table 1; 2026-06-26 for the revision))

This notebook reproduces the manuscript-style Table 1 from `data/BGA_merged_all_20260208_cleaned.csv` and outputs both:

- a pandas table for inspection in-notebook
- a LaTeX table ready for manuscript use

For manuscript Table 1, the RBANS rows use the recorded RBANS index-score columns in the cleaned cohort file rather than the March 16 analysis-ready RBANS summary-composite workflow used elsewhere in the standalone pipeline.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import Markdown, display

DATA_CANDIDATES = [
    Path("data/BGA_merged_all_20260208_cleaned.csv"),
    Path("../data/BGA_merged_all_20260208_cleaned.csv"),
]

TEXT_COLUMNS = ["Subject", "Gender", "Mothertounge", "HandPref", "TestAdmin", "Group", "IBStype"]
BIS_ITEMS = [f"BIS_Q{i}_BL" for i in range(1, 7)]
MEASURE_COLUMNS = [
    "TestAge",
    "Education",
    *BIS_ITEMS,
    "CPT_Detectability",
    "CPT_Omissions",
    "CPT_Commissions",
    "CPT_HRT",
    "TFS_Chalder",
    "HADS_Anxiety",
    "HADS_Depression",
    "RBANS_Memory_Index",
    "RBANS_Visuoaspatial_Index",
    "RBANS_Verbalskills_Index",
    "RBANS_Attention_Index",
    "RBANS_Recall_Index",
    "RBANS_Fullscale",
]
MEASURE_SPECS = [
    ("Age (years)", "TestAge"),
    ("Education (years)", "Education"),
    ("BIS total", "BIS_total"),
    ("Detectability", "CPT_Detectability"),
    ("Omissions", "CPT_Omissions"),
    ("Commissions", "CPT_Commissions"),
    ("HRT", "CPT_HRT"),
    ("Chalder total", "Chalder_total"),
    ("Anxiety", "HADS_Anxiety"),
    ("Depression", "HADS_Depression"),
    ("Immediate Memory", "RBANS_Memory_Index"),
    ("Visuospatial", "RBANS_Visuoaspatial_Index"),
    ("Language", "RBANS_Verbalskills_Index"),
    ("Attention", "RBANS_Attention_Index"),
    ("Delayed Memory", "RBANS_Recall_Index"),
    ("Total Scale", "RBANS_Fullscale"),
]


def resolve_data_path() -> Path:
    for candidate in DATA_CANDIDATES:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not locate BGA_merged_all_20260208_cleaned.csv from the current working directory.")


def load_cleaned_cohort(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";")
    df.columns = df.columns.str.strip()

    for col in TEXT_COLUMNS:
        cleaned = df[col].astype(str).str.strip()
        df[col] = cleaned.mask(cleaned.eq("")).replace("nan", np.nan)

    for col in MEASURE_COLUMNS:
        df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")

    df["BIS_total"] = df[BIS_ITEMS].sum(axis=1, min_count=1)
    df["Chalder_total"] = df["TFS_Chalder"]
    return df


def female_mask(series: pd.Series) -> pd.Series:
    return series.astype(str).str.upper().str.startswith("F")


def cohens_d_pooled(x: pd.Series, y: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    y = pd.to_numeric(y, errors="coerce").dropna()
    nx, ny = len(x), len(y)
    sx, sy = x.std(ddof=1), y.std(ddof=1)
    sp = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))
    return float((x.mean() - y.mean()) / sp)


def summarize_measure(ibs: pd.DataFrame, hc: pd.DataFrame, label: str, col: str) -> dict:
    ibs_vals = pd.to_numeric(ibs[col], errors="coerce").dropna()
    hc_vals = pd.to_numeric(hc[col], errors="coerce").dropna()
    test = stats.ttest_ind(ibs_vals, hc_vals, equal_var=False, nan_policy="omit")
    return {
        "Measure": label,
        "IBS_n": len(ibs_vals),
        "HC_n": len(hc_vals),
        "IBS_mean": ibs_vals.mean(),
        "IBS_sd": ibs_vals.std(ddof=1),
        "HC_mean": hc_vals.mean(),
        "HC_sd": hc_vals.std(ddof=1),
        "t": float(test.statistic),
        "p": float(test.pvalue),
        "d": cohens_d_pooled(ibs_vals, hc_vals),
    }


def p_stars(p: float) -> str:
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def fmt_signed(value: float, digits: int = 2) -> str:
    rounded = round(float(value), digits)
    return f"{rounded:.{digits}f}"


def fmt_mean_sd(mean: float, sd: float) -> str:
    return f"{mean:.1f} ({sd:.1f})"


DATA_PATH = resolve_data_path()
df = load_cleaned_cohort(DATA_PATH)
ibs = df[df["Group"] == "IBS"].copy()
hc = df[df["Group"] == "HC"].copy()

table1_stats = pd.DataFrame([summarize_measure(ibs, hc, label, col) for label, col in MEASURE_SPECS])

female_ibs = int(female_mask(ibs["Gender"]).sum())
female_hc = int(female_mask(hc["Gender"]).sum())
chi2, chi2_p, _, _ = stats.chi2_contingency([
    [female_ibs, len(ibs) - female_ibs],
    [female_hc, len(hc) - female_hc],
], correction=False)  # uncorrected Pearson chi-square (consistent with Cramer's V)

cohort_summary = {
    "IBS_n": len(ibs),
    "HC_n": len(hc),
    "female_pct_ibs": round(female_ibs / len(ibs) * 100, 1),
    "female_pct_hc": round(female_hc / len(hc) * 100, 1),
    "gender_chi2": float(chi2),
    "gender_p": float(chi2_p),
}

cohort_summary

## Effect sizes with 95% confidence intervals

Reproduces the bracketed effect-size CIs reported in manuscript Table 1:

- **Cohen's _d_** (pooled _SD_) with a large-sample **Hedges & Olkin** normal-approximation 95% CI.
- **Cramér's _V_** for the categorical sex comparison, computed from the **uncorrected Pearson** χ², with a **noncentral χ² (Smithson)** 95% CI.

The cell prints both a readable summary and the exact `[low, high]` strings used in the LaTeX table, so the CIs regenerate from code rather than being entered by hand.

In [ ]:
from scipy.optimize import brentq


def cohens_d_ci(x, y, conf=0.95):
    """Cohen's d (pooled SD) with a large-sample Hedges & Olkin 95% CI."""
    x = pd.to_numeric(x, errors="coerce").dropna()
    y = pd.to_numeric(y, errors="coerce").dropna()
    n1, n2 = len(x), len(y)
    sp = np.sqrt(((n1 - 1) * x.std(ddof=1) ** 2 + (n2 - 1) * y.std(ddof=1) ** 2) / (n1 + n2 - 2))
    d = (x.mean() - y.mean()) / sp
    se = np.sqrt((n1 + n2) / (n1 * n2) + d ** 2 / (2 * (n1 + n2)))
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    return float(d), float(d - z * se), float(d + z * se)


def cramers_v_ci(table, conf=0.95):
    """Cramer's V from the uncorrected Pearson chi-square, with a
    noncentral chi-square (Smithson) confidence interval."""
    table = np.asarray(table, dtype=float)
    chi2_obs = stats.chi2_contingency(table, correction=False)[0]
    n = table.sum()
    dof = (table.shape[0] - 1) * (table.shape[1] - 1)
    k = min(table.shape) - 1
    V = np.sqrt(chi2_obs / (n * k))
    alpha = 1 - conf

    def ncp(q):
        if stats.ncx2.cdf(chi2_obs, dof, 0.0) < q:
            return 0.0
        hi = 1.0
        while stats.ncx2.cdf(chi2_obs, dof, hi) > q:
            hi *= 2
        return brentq(lambda lam: stats.ncx2.cdf(chi2_obs, dof, lam) - q, 0.0, hi)

    lam_lo, lam_hi = ncp(1 - alpha / 2), ncp(alpha / 2)
    return float(V), float(np.sqrt(lam_lo / (n * k))), float(np.sqrt(lam_hi / (n * k))), float(chi2_obs)


print("Effect size [95% CI] -- reproduces manuscript Table 1\n")
for label, col in MEASURE_SPECS:
    d, lo, hi = cohens_d_ci(ibs[col], hc[col])
    print(f"{label:18s} d = {d:+.2f} [{lo:+.2f}, {hi:+.2f}]")

sex_table = [
    [female_ibs, len(ibs) - female_ibs],
    [female_hc, len(hc) - female_hc],
]
V, v_lo, v_hi, chi2_unc = cramers_v_ci(sex_table)
print(
    f"\nFemale (sex): uncorrected Pearson chi2 = {chi2_unc:.2f}, "
    f"Cramer's V = {V:.2f} [{v_lo:.2f}, {v_hi:.2f}]"
)

In [ ]:
def row_lookup(label: str) -> pd.Series:
    row = table1_stats.loc[table1_stats["Measure"] == label]
    if row.empty:
        raise KeyError(label)
    return row.iloc[0]


pandas_rows = [
    {
        "Measure": "Demographics",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": "",
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": "",
        "t / chi^2": "",
        "Cohen's d": "",
    },
    {
        "Measure": "Age (years)",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": fmt_mean_sd(row_lookup("Age (years)")["IBS_mean"], row_lookup("Age (years)")["IBS_sd"]),
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": fmt_mean_sd(row_lookup("Age (years)")["HC_mean"], row_lookup("Age (years)")["HC_sd"]),
        "t / chi^2": fmt_signed(row_lookup("Age (years)")["t"]),
        "Cohen's d": fmt_signed(row_lookup("Age (years)")["d"]),
    },
    {
        "Measure": "Education (years)^a",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": fmt_mean_sd(row_lookup("Education (years)")["IBS_mean"], row_lookup("Education (years)")["IBS_sd"]),
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": fmt_mean_sd(row_lookup("Education (years)")["HC_mean"], row_lookup("Education (years)")["HC_sd"]),
        "t / chi^2": fmt_signed(row_lookup("Education (years)")["t"]),
        "Cohen's d": fmt_signed(row_lookup("Education (years)")["d"]),
    },
    {
        "Measure": "Female (%)",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": f"{cohort_summary['female_pct_ibs']:.1f}",
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": f"{cohort_summary['female_pct_hc']:.1f}",
        "t / chi^2": f"{cohort_summary['gender_chi2']:.2f}\u2020",
        "Cohen's d": "--",
    },
    {
        "Measure": "Sleep^b",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": "",
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": "",
        "t / chi^2": "",
        "Cohen's d": "",
    },
    {
        "Measure": "BIS total (0-42)",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": fmt_mean_sd(row_lookup("BIS total")["IBS_mean"], row_lookup("BIS total")["IBS_sd"]),
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": fmt_mean_sd(row_lookup("BIS total")["HC_mean"], row_lookup("BIS total")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('BIS total')['t'])}{p_stars(row_lookup('BIS total')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("BIS total")["d"]),
    },
    {
        "Measure": "Attention (CPT-3 T-scores)^c",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": "",
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": "",
        "t / chi^2": "",
        "Cohen's d": "",
    },
]

for label in ["Detectability", "Omissions", "Commissions", "HRT"]:
    row = row_lookup(label)
    display_label = "Detectability (d')" if label == "Detectability" else label
    pandas_rows.append(
        {
            "Measure": display_label,
            f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": fmt_mean_sd(row["IBS_mean"], row["IBS_sd"]),
            f"HC (n = {cohort_summary['HC_n']})\nM (SD)": fmt_mean_sd(row["HC_mean"], row["HC_sd"]),
            "t / chi^2": f"{fmt_signed(row['t'])}{p_stars(row['p'])}",
            "Cohen's d": fmt_signed(row["d"]),
        }
    )

pandas_rows.extend([
    {
        "Measure": "Fatigue^d",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": "",
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": "",
        "t / chi^2": "",
        "Cohen's d": "",
    },
    {
        "Measure": "Chalder total (0-11)",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": fmt_mean_sd(row_lookup("Chalder total")["IBS_mean"], row_lookup("Chalder total")["IBS_sd"]),
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": fmt_mean_sd(row_lookup("Chalder total")["HC_mean"], row_lookup("Chalder total")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('Chalder total')['t'])}{p_stars(row_lookup('Chalder total')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("Chalder total")["d"]),
    },
    {
        "Measure": "Emotional distress (HADS)^e",
        f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": "",
        f"HC (n = {cohort_summary['HC_n']})\nM (SD)": "",
        "t / chi^2": "",
        "Cohen's d": "",
    },
])

for label in ["Anxiety", "Depression", "Immediate Memory", "Visuospatial", "Language", "Attention", "Delayed Memory", "Total Scale"]:
    if label == "Immediate Memory":
        pandas_rows.append(
            {
                "Measure": "Neurocognition (RBANS index scores)^c",
                f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": "",
                f"HC (n = {cohort_summary['HC_n']})\nM (SD)": "",
                "t / chi^2": "",
                "Cohen's d": "",
            }
        )
    row = row_lookup(label)
    measure_label = {
        "Anxiety": "Anxiety (0-21)",
        "Depression": "Depression (0-21)",
    }.get(label, label)
    pandas_rows.append(
        {
            "Measure": measure_label,
            f"IBS (n = {cohort_summary['IBS_n']})\nM (SD)": fmt_mean_sd(row["IBS_mean"], row["IBS_sd"]),
            f"HC (n = {cohort_summary['HC_n']})\nM (SD)": fmt_mean_sd(row["HC_mean"], row["HC_sd"]),
            "t / chi^2": f"{fmt_signed(row['t'])}{p_stars(row['p'])}",
            "Cohen's d": fmt_signed(row["d"]),
        }
    )

pandas_table = pd.DataFrame(pandas_rows)
pandas_table

| Measure | IBS (n = 65)<br>M (SD) | HC (n = 40)<br>M (SD) | t / chi^2 | Cohen's d |
|---|---:|---:|---:|---:|
| Demographics |  |  |  |  |
| Age (years) | 37.8 (11.4) | 35.6 (12.5) | 0.90 | 0.19 |
| Education (years)^a | 15.8 (1.9) | 16.4 (1.4) | -1.54 | -0.31 |
| Female (%) | 78.5 | 67.5 | 1.04+ | -- |
| Sleep^b |  |  |  |  |
| BIS total (0-42) | 17.6 (7.6) | 10.3 (6.9) | 4.89*** | 0.99 |
| Attention (CPT-3 T-scores)^c |  |  |  |  |
| Detectability (d') | 48.9 (7.8) | 44.3 (7.1) | 3.00** | 0.60 |
| Omissions | 47.4 (6.4) | 45.0 (1.6) | 2.79** | 0.45 |
| Commissions | 51.0 (9.1) | 48.0 (8.1) | 1.70 | 0.34 |
| HRT | 48.4 (8.0) | 48.7 (8.8) | -0.18 | -0.04 |
| Fatigue^d |  |  |  |  |
| Chalder total (0-11) | 6.4 (3.4) | 1.6 (2.5) | 7.37*** | 1.55 |
| Emotional distress (HADS)^e |  |  |  |  |
| Anxiety (0-21) | 8.1 (4.2) | 4.2 (3.3) | 4.97*** | 1.00 |
| Depression (0-21) | 4.7 (3.1) | 2.1 (2.3) | 4.52*** | 0.90 |
| Neurocognition (RBANS index scores)^c |  |  |  |  |
| Immediate Memory | 89.6 (16.1) | 98.3 (15.7) | -2.65** | -0.54 |
| Visuospatial | 94.0 (11.7) | 94.6 (12.4) | -0.24 | -0.05 |
| Language | 98.2 (14.1) | 100.9 (13.9) | -0.97 | -0.20 |
| Attention | 92.5 (12.2) | 100.0 (18.3) | -2.22* | -0.51 |
| Delayed Memory | 93.2 (15.0) | 101.8 (19.8) | -2.30* | -0.51 |
| Total Scale | 90.5 (13.2) | 99.2 (12.9) | -3.25** | -0.67 |

If you want, I can also turn this into a publication-style Markdown table with grouped section rows bolded.

In [ ]:
def latex_signed(value: float, digits: int = 2) -> str:
    rounded = round(float(value), digits)
    if rounded < 0:
        return f"$-{abs(rounded):.{digits}f}$"
    return f"{rounded:.{digits}f}"


def latex_row(label: str, measure_label: str, left_suffix: str = "", use_d: bool = True) -> str:
    row = row_lookup(measure_label)
    test_part = f"{latex_signed(row['t'])}{p_stars(row['p'])}"
    d_part = latex_signed(row["d"]) if use_d else "--"
    return (
        f"\\quad {label}{left_suffix} & "
        f"{fmt_mean_sd(row['IBS_mean'], row['IBS_sd'])} & "
        f"{fmt_mean_sd(row['HC_mean'], row['HC_sd'])} & "
        f"{test_part} & {d_part} \\\\"
    )


latex_lines = [
    "\\begin{table}[htbp]",
    "\\centering",
    "\\caption{Cohort demographics and neuropsychological domain scores by group.}",
    "\\label{tab:table1_20260316}",
    "\\begin{tabular}{lcccc}",
    "\\toprule",
    f"Measure & IBS ($n = {cohort_summary['IBS_n']}$) & HC ($n = {cohort_summary['HC_n']}$) & $t$ / $\\chi^2$ & Cohen's $d$ \\\\",
    " & $M$ ($SD$) & $M$ ($SD$) &  &  \\\\",
    "\\midrule",
    "\\textit{Demographics} &  &  &  &  \\\\",
    latex_row("Age (years)", "Age (years)"),
    latex_row("Education (years)", "Education (years)", left_suffix="$^{a}$"),
    f"\\quad Female (\\%) & {cohort_summary['female_pct_ibs']:.1f} & {cohort_summary['female_pct_hc']:.1f} & {cohort_summary['gender_chi2']:.2f}$^{{\\dagger}}$ & -- \\\\",
    "\\midrule",
    "\\textit{Sleep}$^{b}$ &  &  &  &  \\\\",
    latex_row("BIS total (0--42)", "BIS total"),
    "\\midrule",
    "\\textit{Attention (CPT-3 T-scores)}$^{c}$ &  &  &  &  \\\\",
    latex_row("Detectability ($d'$)", "Detectability"),
    latex_row("Omissions", "Omissions"),
    latex_row("Commissions", "Commissions"),
    latex_row("HRT", "HRT"),
    "\\midrule",
    "\\textit{Fatigue}$^{d}$ &  &  &  &  \\\\",
    latex_row("Chalder total (0--11)", "Chalder total"),
    "\\midrule",
    "\\textit{Emotional distress (HADS)}$^{e}$ &  &  &  &  \\\\",
    latex_row("Anxiety (0--21)", "Anxiety"),
    latex_row("Depression (0--21)", "Depression"),
    "\\midrule",
    "\\textit{Neurocognition (RBANS index scores)}$^{c}$ &  &  &  &  \\\\",
    latex_row("Immediate Memory", "Immediate Memory"),
    latex_row("Visuospatial", "Visuospatial"),
    latex_row("Language", "Language"),
    latex_row("Attention", "Attention"),
    latex_row("Delayed Memory", "Delayed Memory"),
    latex_row("Total Scale", "Total Scale"),
    "\\bottomrule",
    "\\multicolumn{5}{p{0.97\\linewidth}}{\\footnotesize $^{\\dagger}\\chi^2$ test for gender; all others are independent-samples Welch's $t$-tests. *$p < .05$; **$p < .01$; ***$p < .001$. Sample sizes vary by measure due to missing data: $^{a}$HC $n = 28$; $^{b}$IBS $n = 58$, HC $n = 39$; $^{c}$HC $n = 37$; $^{d}$IBS $n = 49$, HC $n = 35$; $^{e}$IBS $n = 57$, HC $n = 36$. Negative $d$ values indicate IBS $<$ HC.}",
    "\\end{tabular}",
    "\\end{table}",
]

latex_table = "\n".join(latex_lines)

print(latex_table)
display(Markdown("```latex\n" + latex_table + "\n```"))

```latex
\begin{table}[htbp]
\centering
\caption{Cohort demographics and neuropsychological domain scores by group.}
\label{tab:table1_20260316}
\begin{tabular}{lcccc}
\toprule
Measure & IBS ($n = 65$) & HC ($n = 40$) & $t$ / $\chi^2$ & Cohen's $d$ \\
 & $M$ ($SD$) & $M$ ($SD$) &  &  \\
\midrule
\textit{Demographics} &  &  &  &  \\
\quad Age (years) & 37.8 (11.4) & 35.6 (12.5) & 0.90 & 0.19 \\
\quad Education (years)$^{a}$ & 15.8 (1.9) & 16.4 (1.4) & $-1.54$ & $-0.31$ \\
\quad Female (\%) & 78.5 & 67.5 & 1.04$^{\dagger}$ & -- \\
\midrule
\textit{Sleep}$^{b}$ &  &  &  &  \\
\quad BIS total (0--42) & 17.6 (7.6) & 10.3 (6.9) & 4.89*** & 0.99 \\
\midrule
\textit{Attention (CPT-3 T-scores)}$^{c}$ &  &  &  &  \\
\quad Detectability ($d'$) & 48.9 (7.8) & 44.3 (7.1) & 3.00** & 0.60 \\
\quad Omissions & 47.4 (6.4) & 45.0 (1.6) & 2.79** & 0.45 \\
\quad Commissions & 51.0 (9.1) & 48.0 (8.1) & 1.70 & 0.34 \\
\quad HRT & 48.4 (8.0) & 48.7 (8.8) & $-0.18$ & $-0.04$ \\
\midrule
\textit{Fatigue}$^{d}$ &  &  &  &  \\
\quad Chalder total (0--11) & 6.4 (3.4) & 1.6 (2.5) & 7.37*** & 1.55 \\
\midrule
\textit{Emotional distress (HADS)}$^{e}$ &  &  &  &  \\
\quad Anxiety (0--21) & 8.1 (4.2) & 4.2 (3.3) & 4.97*** & 1.00 \\
\quad Depression (0--21) & 4.7 (3.1) & 2.1 (2.3) & 4.52*** & 0.90 \\
\midrule
\textit{Neurocognition (RBANS index scores)}$^{c}$ &  &  &  &  \\
\quad Immediate Memory & 89.6 (16.1) & 98.3 (15.7) & $-2.65$** & $-0.54$ \\
\quad Visuospatial & 94.0 (11.7) & 94.6 (12.4) & $-0.24$ & $-0.05$ \\
\quad Language & 98.2 (14.1) & 100.9 (13.9) & $-0.97$ & $-0.20$ \\
\quad Attention & 92.5 (12.2) & 100.0 (18.3) & $-2.22$* & $-0.51$ \\
\quad Delayed Memory & 93.2 (15.0) & 101.8 (19.8) & $-2.30$* & $-0.51$ \\
\quad Total Scale & 90.5 (13.2) & 99.2 (12.9) & $-3.25$** & $-0.67$ \\
\bottomrule
\multicolumn{5}{p{0.97\linewidth}}{\footnotesize $^{\dagger}\chi^2$ test for gender; all others are independent-samples Welch's $t$-tests. *$p < .05$; **$p < .01$; ***$p < .001$. Sample sizes vary by measure due to missing data: $^{a}$HC $n = 28$; $^{b}$IBS $n = 58$, HC $n = 39$; $^{c}$HC $n = 37$; $^{d}$IBS $n = 49$, HC $n = 35$; $^{e}$IBS $n = 57$, HC $n = 36$. Negative $d$ values indicate IBS $<$ HC.}
\end{tabular}
\end{table}
```